In [34]:
import pandas as pd
import os
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder,MinMaxScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

In [40]:
df=pd.read_csv("Ecommerce.csv", on_bad_lines='skip')
df=df.drop(columns=["CID","TID","Purchase Date"])
x=df.drop(columns="Discount Amount (INR)",axis=1)
y=df["Discount Amount (INR)"]
cat_col=x.select_dtypes(include=object)
num_col=x.select_dtypes(exclude=object)

/tmp/ipykernel_540/1143097229.py:1: DtypeWarning: Columns (0,1) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv("Ecommerce.csv", on_bad_lines='skip')


In [41]:
num_pipe=Pipeline(steps=[
("imputer", SimpleImputer(strategy="median")),
("scaler",MinMaxScaler())
])

In [44]:
cat_pipe=Pipeline(steps=[
("imputer",SimpleImputer(strategy="most_frequent")),
("encoder",OneHotEncoder(drop="first",handle_unknown="ignore"))
])

In [45]:
preprocessor=ColumnTransformer([
("num",num_pipe,num_col),
("cat",cat_pipe,cat_col)
])

In [ ]:
# Decision Tree Classifier
model=Pipeline(steps=[
("preprocessor" , preprocessor),
("tree",DecisionTreeClassifier(random_state=42))
])

In [ ]:
# Random Forest Classifier
model=Pipeline(steps=[
("preprocessor" , preprocessor),
("forest",RandomForestClassifier(random_state=42))
])

In [50]:
# Logistic Regression
model=Pipeline(steps=[
("preprocessor" , preprocessor),
("linear",LinearRegression())
])

In [ ]:
# Fix the column specifications for the preprocessor
clean_num_cols = num_col.columns.tolist()
clean_cat_cols = cat_col.columns.tolist()

# Re-build the preprocessor with correct column specifications
preprocessor = ColumnTransformer([
    ("num", num_pipe, clean_num_cols),
    ("cat", cat_pipe, clean_cat_cols)
])

# Re-build and fit the model
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("linear", LinearRegression())
])

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.20)
model.fit(x_train,y_train)

In [ ]:
new_df=pd.read_excel("EcommercePrediction.csv")
new_df=new_df.drop(columns=["CID","TID","Purchase Date","Discount Amount (INR)"])
prediction=model.predict(new_df)
new_df["predict_output"]=prediction
new_df.to_csv("output_LR.csv",index=False)